# Topics from news headlines

Group related documents and give each group a topic name. No labels or quantitative model evaluation are needed. Read the activity README for the complete contracts and scoring. Add your own code cells after the prompts, before the assertions; preserve supplied cells and IDs.


In [ ]:
from pathlib import Path
import csv
import numpy as np
from scipy.sparse import issparse
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.cluster import KMeans, MeanShift

activity_dir = Path.cwd() if (Path.cwd() / "data/headlines.csv").exists() else Path.cwd().parent
with (activity_dir / "data/headlines.csv").open(newline="", encoding="utf-8") as file:
    headlines = [row["headline"] for row in csv.DictReader(file)]
print(f"Loaded {len(headlines)} synthetic headlines")
print(*headlines[:3], sep="\n")


## 1. Vectorize — 20 points

Fit `CountVectorizer(stop_words="english")` on `headlines`. Save `vectorizer`, `counts`, and `terms`. Fit a default `TfidfTransformer`, saving `tfidf` and `X`. Keep the supplied row order. Add your code cell below.


In [ ]:
assert isinstance(vectorizer, CountVectorizer)
assert vectorizer.stop_words == "english"
assert issparse(counts) and counts.shape == (len(headlines), len(terms))
expected_vectorizer = CountVectorizer(stop_words="english")
expected_counts = expected_vectorizer.fit_transform(headlines)
assert np.array_equal(terms, expected_vectorizer.get_feature_names_out())
assert (counts != expected_counts).nnz == 0
assert isinstance(tfidf, TfidfTransformer)
assert issparse(X) and X.shape == counts.shape
assert np.allclose(X.toarray(), TfidfTransformer().fit_transform(counts).toarray())
assert np.allclose(vectorizer.transform(headlines).toarray(), counts.toarray())
assert np.allclose(tfidf.transform(counts).toarray(), X.toarray())


## 2. Cluster — 30 points

Fit `KMeans(n_clusters=4, random_state=42, n_init=10)` on `X`; save `kmeans` and `kmeans_labels`. Choose a positive `bandwidth` (start with `0.9`) and fit `MeanShift(bandwidth=bandwidth)` on `X.toarray()`; save `meanshift` and `meanshift_labels`. Set `labels_by_method = {"kmeans": kmeans_labels, "meanshift": meanshift_labels}`. Print group counts and group sizes for both methods.

MeanShift needs dense input in this example. It has no `n_clusters` argument, but bandwidth changes the grouping. One group or many groups are valid findings. No tuning or quality threshold is required. Add your code cell below.


In [ ]:
assert isinstance(kmeans, KMeans) and kmeans.n_clusters == 4
assert kmeans.random_state == 42 and kmeans.n_init == 10
assert isinstance(meanshift, MeanShift) and bandwidth > 0
assert meanshift.bandwidth == bandwidth
assert set(labels_by_method) == {"kmeans", "meanshift"}
for name, model, labels in [("kmeans", kmeans, kmeans_labels), ("meanshift", meanshift, meanshift_labels)]:
    assert np.asarray(labels).shape == (len(headlines),)
    assert np.issubdtype(np.asarray(labels).dtype, np.integer)
    assert np.array_equal(labels_by_method[name], labels)
    assert np.array_equal(labels, model.labels_)
    assert np.array_equal(labels, model.predict(X if name == "kmeans" else X.toarray()))
assert len(np.unique(kmeans_labels)) == 4


## 3. Common words and examples — 25 points

Create `cluster_summaries[method][cluster_id]` for every group of both methods, using integer IDs. Each entry contains `size`, `top_words`, and `examples`. Sum the group's rows of **counts**, discard zero frequencies, sort by descending frequency and then alphabetically, and keep the first five words (or all if fewer). Keep the first three original headlines in row order (or all if fewer). Print every summary. Frequency and TF-IDF weight are different quantities. Add your code cell below.


In [ ]:
assert set(cluster_summaries) == set(labels_by_method)
for method, labels in labels_by_method.items():
    assert set(cluster_summaries[method]) == set(map(int, np.unique(labels)))
    for cluster_id, summary in cluster_summaries[method].items():
        rows = np.flatnonzero(np.asarray(labels) == cluster_id)
        frequencies = np.asarray(counts[rows].sum(axis=0)).ravel()
        indices = sorted(np.flatnonzero(frequencies > 0), key=lambda i: (-frequencies[i], terms[i]))[:5]
        assert summary["size"] == len(rows)
        assert summary["top_words"] == [str(terms[i]) for i in indices]
        assert summary["examples"] == [headlines[i] for i in rows[:3]]


## 4. Interpret the findings — 25 points

Create `topic_names[method][cluster_id]` with a short, manually chosen topic name for every group. You may name a group "mixed" or "unclear". Write `reflection`, a paragraph of at least 80 words, in Spanish or English. Compare both methods; support topic names with common words and example headlines; discuss at least one mixed or ambiguous headline; and explain the difference between choosing `n_clusters` and choosing `bandwidth`. Print the names and reflection.

The teacher reviews the reasoning; there is no single correct set of names. The assertions below only check completeness. Add your code cell below.


In [ ]:
assert set(topic_names) == set(labels_by_method)
for method, labels in labels_by_method.items():
    assert set(topic_names[method]) == set(map(int, np.unique(labels)))
    assert all(isinstance(name, str) and name.strip() for name in topic_names[method].values())
assert isinstance(reflection, str) and len(reflection.split()) >= 80
